In [0]:
# --- CELDA 1: SETUP ---
%pip install scikit-learn pandas --quiet
dbutils.library.restartPython()

In [0]:
# --- CELDA 2: RUTAS (Tu código robusto) ---
import sys, os
notebook_path = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_path, ".."))
FOLDER_NAME = "src" 

src_path = os.path.join(project_root, FOLDER_NAME)
if not os.path.exists(src_path): src_path = os.path.join(notebook_path, FOLDER_NAME)
if src_path not in sys.path: sys.path.append(src_path)


In [0]:
# --- CELDA 3: IMPORTS ---
# Importamos el archivo que acabamos de crear
from nombre_paquete.preprocessing import transformers
import pandas as pd

# Autoreload para desarrollo rápido
%load_ext autoreload
%autoreload 2


In [0]:
# --- CELDA 4: CARGAR DATOS (Bronze Layer) ---
table_input = "climate_data_raw"
print(f"📥 Leyendo datos desde: {table_input}...")

try:
    # Leemos la tabla Delta guardada en el Notebook 01
    df_raw = spark.table(table_input).toPandas()
    print(f" Datos cargados exitosamente. Filas: {len(df_raw)}")
except Exception as e:
    print(f" Error: La tabla '{table_input}' no existe. Ejecuta el Notebook 01 primero.")


In [0]:

# --- CELDA 5: EJECUTAR LIMPIEZA ---
# Llamamos a la función 1 de tu script
df_clean = transformers.preprocess_data(df_raw)

# Verificación rápida
print("Muestra de datos procesados:")
display(df_clean.head())

In [0]:
TARGET = 'energy_consumption' # Tu variable objetivo

# Llamamos a la función corregida
# Ahora train_df y test_df tendrán el consumo en VALORES REALES (miles/millones), no en decimales.
train_df, test_df, scaler_model = transformers.split_and_scale(
    df_clean, 
    target_col=TARGET, 
    cutoff_date='2024-01-01'
)

# Verifica mirando los datos. La columna 'energy_consumption' debería tener números grandes.
display(train_df[[TARGET]].head())


In [0]:
# Clean column names to remove invalid characters
def clean_column_names(df):
    df.columns = [
        col.strip()
        .replace(' ', '_')
        .replace(',', '')
        .replace(';', '')
        .replace('{', '')
        .replace('}', '')
        .replace('(', '')
        .replace(')', '')
        .replace('\n', '')
        .replace('\t', '')
        .replace('=', '')
        for col in df.columns
    ]
    return df

train_save = clean_column_names(train_df.reset_index())
test_save = clean_column_names(test_df.reset_index())

print(" Guardando tablas procesadas en el Data Lake...")

spark.createDataFrame(train_save)\
    .write.format("delta").mode("overwrite").saveAsTable("climate_train_silver")

spark.createDataFrame(test_save)\
    .write.format("delta").mode("overwrite").saveAsTable("climate_test_silver")

print(" Éxito: Tablas 'climate_train_silver' y 'climate_test_silver' creadas.")

In [0]:
# --- CELDA: CORRELACIÓN ENTRE VARIABLES Y TARGET ---
import matplotlib.pyplot as plt
import seaborn as sns

TARGET_COL = "energy_consumption"  # Cambia esto por el nombre real de tu variable objetivo

# Correlación entre todas las variables
corr_matrix = df_clean.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Matriz de correlación (heatmap)")
plt.tight_layout()
display(plt.gcf())
plt.close()

# Correlación de cada variable con el target
if TARGET_COL in df_clean.columns:
    corr_target = corr_matrix[TARGET_COL].drop(TARGET_COL).sort_values()
    plt.figure(figsize=(10, 6))
    sns.barplot(x=corr_target.values, y=corr_target.index, palette="vlag")
    plt.title(f"Correlación de variables con '{TARGET_COL}'")
    plt.xlabel("Coeficiente de correlación")
    plt.ylabel("Variable")
    plt.tight_layout()
    display(plt.gcf())
    plt.close()
else:
    print(f"La columna '{TARGET_COL}' no está en el DataFrame.")